# 01 Exploration: Voice Periodicity (Summary)

**Question:** Do any voice features appear to fluctuate periodically in alignment with menstrual-cycle timing?

## Methodology

- Data source: `data/processed/analysis_first_pass_merged.parquet`
- Scope: all daily voice features (`vowel_egemaps_*`, `prosody_egemaps_*`) with cycle context (`cycle_day`, `phase_label`, `cycle_week`, `cycle_start_date`)
- Inclusion thresholds:
  - at least 30 non-null observations per feature
  - at least 8 observations in each phase group (`follicular`, `luteal`)
  - at least 5 observations per included cycle-week group
- Per-feature cycle-alignment signals:
  - phase separation: Mann-Whitney U p-value + absolute Cliff's delta
  - cycle-week differences: Kruskal-Wallis p-value + epsilon-squared effect size
- Ranking score:
  - `periodicity_score = phase_effect + week_effect + 0.25 * (-log10(phase_p) + -log10(week_p))`
  - higher score indicates stronger evidence of cycle-aligned fluctuation

This is exploratory ranking for candidate discovery, not confirmatory inference.

## Results

Run summary:

- rows: `176`
- voice features scanned: `176`
- cycle columns used: `date`, `cycle_day`, `phase_label`, `cycle_week`, `cycle_start_date`
- ranked features: `176`

Top 8 periodicity candidates:

1. `prosody_egemaps_mfcc2_sma3_stddevNorm` — score `1.6448`, phase p `0.0055`, week p `0.0193`
2. `vowel_egemaps_F2bandwidth_sma3nz_stddevNorm` — score `1.2346`, phase p `0.0119`, week p `0.1026`
3. `prosody_egemaps_logRelF0-H1-H2_sma3nz_amean` — score `1.1890`, phase p `0.0231`, week p `0.0709`
4. `prosody_egemaps_mfcc3_sma3_amean` — score `1.1481`, phase p `0.0307`, week p `0.0671`
5. `prosody_egemaps_F1bandwidth_sma3nz_stddevNorm` — score `1.0429`, phase p `0.0172`, week p `0.2340`
6. `vowel_egemaps_loudness_sma3_stddevNorm` — score `1.0305`, phase p `0.1208`, week p `0.0274`
7. `prosody_egemaps_alphaRatioUV_sma3nz_amean` — score `1.0131`, phase p `0.0815`, week p `0.0515`
8. `prosody_egemaps_mfcc3V_sma3nz_amean` — score `0.9263`, phase p `0.0403`, week p `0.1922`

Across all features:

- features with phase p <= 0.05: `9`
- features with cycle-week p <= 0.05: `17`

## Interpretation

- There is **early evidence** that some voice features vary in a cycle-aligned way.
- The strongest candidates in this pass are mostly prosody-derived features, with a smaller number of vowel features.
- Signal appears in both lenses:
  - follicular vs luteal separation
  - cycle-week differences across the cycle trajectory
- This supports moving to a targeted deep-dive on a short feature list.

Caveats:

- This is exploratory with many simultaneous feature tests.
- Daily voice coverage is limited, so some effects may be unstable.
- Treat this as candidate discovery rather than final statistical confirmation.

Recommended next step:

- Select the top 5-10 features above for focused visual repeatability checks across cycles, then run confirmatory follow-up analyses.

## Longitudinal Visualization: Top Prosody Features

Daily values with 7-day rolling average, starting from Jan 16, 2026 (when recordings became consistent).

Feature set:

1. **MFCC2 Variability** (`prosody_egemaps_mfcc2_sma3_stddevNorm`)
2. **Spectral Balance H1-H2** (`prosody_egemaps_logRelF0-H1-H2_sma3nz_amean`)
3. **MFCC3 Mean** (`prosody_egemaps_mfcc3_sma3_amean`)
4. **F1 Bandwidth Variability** (`prosody_egemaps_F1bandwidth_sma3nz_stddevNorm`)
5. **Alpha Ratio Unvoiced** (`prosody_egemaps_alphaRatioUV_sma3nz_amean`)

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("data/processed/analysis_first_pass_merged.parquet")
FEATURES = [
    "prosody_egemaps_mfcc2_sma3_stddevNorm",
    "prosody_egemaps_logRelF0-H1-H2_sma3nz_amean",
    "prosody_egemaps_mfcc3_sma3_amean",
    "prosody_egemaps_F1bandwidth_sma3nz_stddevNorm",
    "prosody_egemaps_alphaRatioUV_sma3nz_amean",
]


def sort_week_label(value: str) -> int:
    if value.startswith("week_"):
        suffix = value.split("_", maxsplit=1)[1]
        if suffix.isdigit():
            return int(suffix)
    return 999


def build_palette(labels: list[str]) -> dict[str, tuple[float, float, float, float]]:
    cmap = plt.get_cmap("viridis", len(labels))
    return {label: cmap(index) for index, label in enumerate(labels)}


source = pd.read_parquet(DATA_PATH)
plot_df = source[["date", "cycle_week", *FEATURES]].copy()
plot_df["date"] = pd.to_datetime(plot_df["date"], errors="coerce")
plot_df = plot_df.dropna(subset=["date", "cycle_week"])

plot_df = plot_df.melt(
    id_vars=["date", "cycle_week"],
    value_vars=FEATURES,
    var_name="feature",
    value_name="value",
)
plot_df["value"] = pd.to_numeric(plot_df["value"], errors="coerce")
plot_df = plot_df.dropna(subset=["value"]).copy()
plot_df["cycle_week"] = plot_df["cycle_week"].astype(str)

week_order = sorted(plot_df["cycle_week"].unique().tolist(), key=sort_week_label)
palette = build_palette(week_order)

n_features = len(FEATURES)
fig, axes = plt.subplots(n_features, 1, figsize=(16, 3.5 * n_features), sharex=True)
if n_features == 1:
    axes = [axes]

for feature_index, feature in enumerate(FEATURES):
    ax = axes[feature_index]
    subset = plot_df[plot_df["feature"] == feature].sort_values("date")

    # Draw one longitudinal line per cycle-week group on a shared date axis.
    for week_label in week_order:
        group = subset[subset["cycle_week"] == week_label]
        if group.empty:
            continue

        ax.plot(
            group["date"],
            group["value"],
            marker="o",
            markersize=4,
            linewidth=1.5,
            color=palette[week_label],
            alpha=0.9,
            label=week_label,
        )

    ax.set_title(feature)
    ax.set_ylabel("Feature value")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    seen = set()
    unique_handles = []
    unique_labels = []
    for handle, label in zip(handles, labels):
        if label in seen:
            continue
        seen.add(label)
        unique_handles.append(handle)
        unique_labels.append(label)
    axes[0].legend(unique_handles, unique_labels, title="Cycle week", bbox_to_anchor=(1.02, 1), loc="upper left")

axes[-1].set_xlabel("Date")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Outlier Review: Potential Bad Recording Days

This section identifies candidate outliers for the selected prosody features using a robust per-feature rule:

- `robust_z = 0.6745 * (x - median) / MAD`
- outlier threshold: `|robust_z| >= 3.5`

A day is flagged for manual review when either:

- at least 2 selected features are outliers on the same date, or
- any selected feature has `|robust_z| >= 5.0`

These are **review flags**, not automatic deletions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

OUTLIER_Z_THRESHOLD = 3.5
HARD_OUTLIER_Z_THRESHOLD = 5.0


def robust_zscore(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    median = numeric.median()
    mad = (numeric - median).abs().median()

    if pd.isna(mad) or mad == 0:
        return pd.Series(np.nan, index=numeric.index)

    return 0.6745 * (numeric - median) / mad


data_path = DATA_PATH if "DATA_PATH" in globals() else Path("data/processed/analysis_first_pass_merged.parquet")
features = FEATURES if "FEATURES" in globals() else [
    "prosody_egemaps_mfcc2_sma3_stddevNorm",
    "prosody_egemaps_logRelF0-H1-H2_sma3nz_amean",
    "prosody_egemaps_mfcc3_sma3_amean",
    "prosody_egemaps_F1bandwidth_sma3nz_stddevNorm",
    "prosody_egemaps_alphaRatioUV_sma3nz_amean",
]

source = pd.read_parquet(data_path).copy()
source["date"] = pd.to_datetime(source["date"], errors="coerce")

long_df = source[["date", "cycle_week", *features]].melt(
    id_vars=["date", "cycle_week"],
    value_vars=features,
    var_name="feature",
    value_name="value",
)
long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
long_df = long_df.dropna(subset=["date", "value"]).copy()

long_df["robust_z"] = long_df.groupby("feature")["value"].transform(robust_zscore)
long_df["is_outlier"] = long_df["robust_z"].abs() >= OUTLIER_Z_THRESHOLD

outlier_feature_lists = (
    long_df[long_df["is_outlier"]]
    .groupby("date")["feature"]
    .apply(lambda values: ", ".join(sorted(values.unique())))
    .rename("outlier_features")
)

outlier_summary = (
    long_df.groupby("date", as_index=False)
    .agg(
        outlier_feature_count=("is_outlier", "sum"),
        max_abs_robust_z=("robust_z", lambda s: float(np.nanmax(np.abs(s))) if s.notna().any() else np.nan),
    )
    .merge(outlier_feature_lists, on="date", how="left")
)

context_columns = [
    "date",
    "phase_label",
    "cycle_week",
    "cycle_day",
    "has_prosody",
    "prosody_recording_count",
    "prosody_recording_ids",
]
existing_context_columns = [column for column in context_columns if column in source.columns]
day_context = source[existing_context_columns].drop_duplicates(subset=["date"], keep="last")

outlier_review = outlier_summary.merge(day_context, on="date", how="left")
outlier_review["manual_review_flag"] = (
    (outlier_review["outlier_feature_count"] >= 2)
    | (outlier_review["max_abs_robust_z"] >= HARD_OUTLIER_Z_THRESHOLD)
)
outlier_review["low_prosody_support"] = outlier_review["prosody_recording_count"].fillna(0) <= 1
outlier_review["priority_review"] = outlier_review["manual_review_flag"] | (
    (outlier_review["outlier_feature_count"] >= 1) & outlier_review["low_prosody_support"]
)

review_columns = [
    "date",
    "phase_label",
    "cycle_week",
    "cycle_day",
    "outlier_feature_count",
    "max_abs_robust_z",
    "outlier_features",
    "prosody_recording_count",
    "manual_review_flag",
    "priority_review",
]
review_columns = [column for column in review_columns if column in outlier_review.columns]

outlier_review_sorted = outlier_review.sort_values(
    by=["priority_review", "manual_review_flag", "outlier_feature_count", "max_abs_robust_z", "date"],
    ascending=[False, False, False, False, True],
).reset_index(drop=True)

print(f"Rows scanned: {len(source)}")
print(f"Feature-level outlier points: {int(long_df['is_outlier'].sum())}")
print(f"Dates with >=1 outlier feature: {int((outlier_review['outlier_feature_count'] >= 1).sum())}")
print(f"Dates flagged for manual review: {int(outlier_review['manual_review_flag'].sum())}")
print(f"Dates flagged as priority review: {int(outlier_review['priority_review'].sum())}")

outlier_review_sorted.loc[outlier_review_sorted["outlier_feature_count"] >= 1, review_columns].head(30)

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt


if "sort_week_label" not in globals():
    def sort_week_label(value: str) -> int:
        if value.startswith("week_"):
            suffix = value.split("_", maxsplit=1)[1]
            if suffix.isdigit():
                return int(suffix)
        return 999


if "build_palette" not in globals():
    def build_palette(labels: list[str]) -> dict[str, tuple[float, float, float, float]]:
        cmap = plt.get_cmap("viridis", len(labels))
        return {label: cmap(index) for index, label in enumerate(labels)}


plot_long = long_df.copy()
plot_long["cycle_week"] = plot_long["cycle_week"].astype(str)
week_order = sorted(plot_long["cycle_week"].dropna().unique().tolist(), key=sort_week_label)
palette = build_palette(week_order)

fig, axes = plt.subplots(len(features), 1, figsize=(16, 3.8 * len(features)), sharex=True)
if len(features) == 1:
    axes = [axes]

for feature_index, feature in enumerate(features):
    ax = axes[feature_index]
    feature_data = plot_long[plot_long["feature"] == feature].sort_values("date")

    for week_label in week_order:
        week_data = feature_data[feature_data["cycle_week"] == week_label]
        if week_data.empty:
            continue

        ax.plot(
            week_data["date"],
            week_data["value"],
            color=palette[week_label],
            linewidth=1.25,
            alpha=0.75,
            marker="o",
            markersize=3,
            label=week_label,
        )

    outliers = feature_data[feature_data["is_outlier"]]
    ax.scatter(
        outliers["date"],
        outliers["value"],
        color="crimson",
        marker="x",
        s=70,
        linewidth=1.5,
        label="outlier",
        zorder=5,
    )

    # Annotate the most extreme outliers to simplify manual audio review.
    if not outliers.empty:
        top_outliers = outliers.assign(abs_z=outliers["robust_z"].abs()).nlargest(3, "abs_z")
        for _, row in top_outliers.iterrows():
            ax.annotate(
                row["date"].strftime("%Y-%m-%d"),
                xy=(row["date"], row["value"]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
                color="crimson",
            )

    ax.set_title(f"{feature} (outliers marked)")
    ax.set_ylabel("Feature value")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    seen = set()
    unique_handles = []
    unique_labels = []
    for handle, label in zip(handles, labels):
        if label in seen:
            continue
        seen.add(label)
        unique_handles.append(handle)
        unique_labels.append(label)
    axes[0].legend(unique_handles, unique_labels, title="Cycle week / flag", bbox_to_anchor=(1.02, 1), loc="upper left")

axes[-1].set_xlabel("Date")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Longitudinal Smoothing (Cycle-Agnostic)

This view is fully calendar-time and agnostic to cycle labels.

For each selected feature, we show:

- daily observations over `date` (points)
- 7-day rolling average over `date` (line)
- week-from-start averages, where:
  - week 1 = first 7 days from earliest recording date
  - week 2 = next 7 days, etc.

This makes long-term trends easier to read while preserving raw data context.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("data/processed/analysis_first_pass_merged.parquet")
FEATURES = [
    "prosody_egemaps_mfcc2_sma3_stddevNorm",
    "prosody_egemaps_logRelF0-H1-H2_sma3nz_amean",
    "prosody_egemaps_mfcc3_sma3_amean",
    "prosody_egemaps_F1bandwidth_sma3nz_stddevNorm",
    "prosody_egemaps_alphaRatioUV_sma3nz_amean",
]

ROLLING_WINDOW_DAYS = 7

source = pd.read_parquet(DATA_PATH).copy()
source["date"] = pd.to_datetime(source["date"], errors="coerce")
source = source.dropna(subset=["date"]).sort_values("date")

long_df = source[["date", *FEATURES]].melt(
    id_vars=["date"],
    value_vars=FEATURES,
    var_name="feature",
    value_name="value",
)
long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
long_df = long_df.dropna(subset=["value"]).sort_values(["feature", "date"])

origin_date = long_df["date"].min()
long_df["week_from_start"] = ((long_df["date"] - origin_date).dt.days // 7 + 1).astype(int)

# 7-day rolling mean on daily-resampled series.
rolling_rows = []
for feature, feature_df in long_df.groupby("feature"):
    daily_series = feature_df.groupby("date", as_index=True)["value"].mean().sort_index()
    rolling_series = daily_series.rolling(window=ROLLING_WINDOW_DAYS, min_periods=3).mean()
    rolling_part = rolling_series.reset_index().rename(columns={"value": "rolling_mean"})
    rolling_part["feature"] = feature
    rolling_rows.append(rolling_part)

rolling_df = pd.concat(rolling_rows, ignore_index=True)

weekly_df = (
    long_df.groupby(["feature", "week_from_start"], as_index=False)
    .agg(weekly_mean=("value", "mean"), weekly_count=("value", "size"))
)
weekly_df["week_start_date"] = origin_date + pd.to_timedelta((weekly_df["week_from_start"] - 1) * 7, unit="D")

n_features = len(FEATURES)
fig, axes = plt.subplots(n_features, 1, figsize=(16, 4.0 * n_features), sharex=True)
if n_features == 1:
    axes = [axes]

for index, feature in enumerate(FEATURES):
    ax = axes[index]

    raw = long_df[long_df["feature"] == feature]
    roll = rolling_df[rolling_df["feature"] == feature]
    week = weekly_df[weekly_df["feature"] == feature]

    ax.scatter(
        raw["date"],
        raw["value"],
        s=18,
        alpha=0.35,
        color="gray",
        label="daily value",
    )
    ax.plot(
        roll["date"],
        roll["rolling_mean"],
        linewidth=2.2,
        color="royalblue",
        label=f"{ROLLING_WINDOW_DAYS}-day rolling mean",
    )
    ax.plot(
        week["week_start_date"],
        week["weekly_mean"],
        marker="o",
        markersize=4,
        linewidth=1.4,
        color="darkorange",
        label="week-from-start mean",
    )

    ax.set_title(feature)
    ax.set_ylabel("Feature value")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    axes[0].legend(handles, labels, loc="upper left")

axes[-1].set_xlabel("Date")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

weekly_pivot = weekly_df.pivot(index="week_from_start", columns="feature", values="weekly_mean")
weekly_pivot.tail(12)

In [ ]:
import matplotlib.pyplot as plt

week_plot_df = weekly_df.copy()
week_plot_df = week_plot_df[week_plot_df["weekly_count"] >= 1].copy()

fig, axes = plt.subplots(len(FEATURES), 1, figsize=(16, 3.2 * len(FEATURES)), sharex=True)
if len(FEATURES) == 1:
    axes = [axes]

for index, feature in enumerate(FEATURES):
    ax = axes[index]
    feature_week = week_plot_df[week_plot_df["feature"] == feature].sort_values("week_from_start")

    ax.plot(
        feature_week["week_from_start"],
        feature_week["weekly_mean"],
        marker="o",
        linewidth=1.8,
        markersize=4,
        color="darkorange",
    )

    ax.set_title(f"{feature} (weekly mean)")
    ax.set_ylabel("Weekly mean")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Week from first recording date")
plt.tight_layout()
plt.show()